<a href="https://colab.research.google.com/github/2000030914/2000030914/blob/main/draft_optimized_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tensorflow matplotlib seaborn scikit-learn

import tensorflow as tf
import numpy as np
import os
import shutil
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input

REPRODUCIBILITY

In [2]:
tf.random.set_seed(42)
np.random.seed(42)

DATA CLEANING

In [3]:
ROOT_PATH = "/content/drive/MyDrive/AnnotatedUltrasoundLiver_Dataset"
CLEAN_PATH = "/content/LiverDataset"

if os.path.exists(CLEAN_PATH):
    shutil.rmtree(CLEAN_PATH)
os.makedirs(CLEAN_PATH)

print("Cleaning dataset...")

for class_name in os.listdir(ROOT_PATH):
    class_path = os.path.join(ROOT_PATH, class_name)
    image_folder = os.path.join(class_path, "image")

    if os.path.exists(image_folder):
        new_class_path = os.path.join(CLEAN_PATH, class_name)
        os.makedirs(new_class_path, exist_ok=True)

        for img in os.listdir(image_folder):
            if img.lower().endswith(('.png','.jpg','.jpeg')):
                shutil.copy2(os.path.join(image_folder,img),
                             os.path.join(new_class_path,img))

print("Dataset cleaned!")


Cleaning dataset...
Dataset cleaned!


VERIFY DATA

In [4]:
print("\nVerifying dataset...")

total_images = 0
class_counts = {}
corrupted = []

for cls in os.listdir(CLEAN_PATH):
    cls_path = os.path.join(CLEAN_PATH, cls)
    images = os.listdir(cls_path)
    class_counts[cls] = len(images)
    total_images += len(images)

    for img in images:
        try:
            Image.open(os.path.join(cls_path,img)).verify()
        except:
            corrupted.append(img)

print("Total images:", total_images)
print("Corrupted:", len(corrupted))
print("Class distribution:", class_counts)


Verifying dataset...
Total images: 735
Corrupted: 0
Class distribution: {'Benign': 200, 'Normal': 100, 'Malignant': 435}


DATASET LOADING

In [5]:
IMG_SIZE = (224,224)
BATCH_SIZE = 16

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    CLEAN_PATH,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

temp_ds = tf.keras.preprocessing.image_dataset_from_directory(
    CLEAN_PATH,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_size = int(0.5 * len(temp_ds))
val_ds = temp_ds.take(val_size)
test_ds = temp_ds.skip(val_size)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

Found 735 files belonging to 3 classes.
Using 588 files for training.
Found 735 files belonging to 3 classes.
Using 147 files for validation.


DATA AUGMENTATION

In [6]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

CLASS WEIGHTS

In [7]:
labels = []
for _, y in train_ds.unbatch():
    labels.append(y.numpy())

labels = np.array(labels)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights = dict(enumerate(class_weights))
print("Class weights:", class_weights)

Class weights: {0: np.float64(1.2327044025157232), 1: np.float64(0.5681159420289855), 2: np.float64(2.3333333333333335)}


In [8]:
results = {}

models list

In [9]:
from tensorflow.keras.applications import (
    MobileNetV2, ResNet50, DenseNet121,
    EfficientNetB0, EfficientNetV2B0, ConvNeXtTiny, MobileNetV3Small
)

from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.applications.densenet import preprocess_input as dense_pre
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input as effv2_pre
from tensorflow.keras.applications.convnext import preprocess_input as conv_pre
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as mob3_pre

In [10]:
def build_model(base_model, preprocess_fn):

    base_model.trainable = False

    inputs = layers.Input(shape=(224,224,3))

    x = data_augmentation(inputs)
    x = preprocess_fn(x)

    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = models.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model, base_model   # ✅ FIXED

In [ ]:
models_to_train = {
    "MobileNetV2": (MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3)), mob_pre),
    "ResNet50": (ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3)), res_pre),
    "DenseNet121": (DenseNet121(weights='imagenet', include_top=False, input_shape=(224,224,3)), dense_pre),
    "EfficientNetB0": (EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3)), eff_pre),
    "EfficientNetV2B0": (EfficientNetV2B0(weights='imagenet', include_top=False, input_shape=(224,224,3)), effv2_pre),
    "ConvNeXtTiny": (ConvNeXtTiny(weights='imagenet', include_top=False, input_shape=(224,224,3)), conv_pre)
}

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
24274472/24274472 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


FINAL TRAINING LOOP

In [ ]:
results = {}

# Define callbacks before the loop
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath='best_model_{epoch:02d}_{val_accuracy:.2f}.keras',
        save_best_only=True,
        monitor='val_accuracy',
        mode='max',
        verbose=0
    )
]

for name, (base_model, pre_fn) in models_to_train.items():

    print(f"\n🚀 Training {name}...\n")

    model, base_model = build_model(base_model, pre_fn)

    # -------- Stage 1 --------
    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=10,
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=1
    )

    # -------- Stage 2 (Fine-tuning) --------
    print(f"\n🔧 Fine-tuning {name}...\n")

    base_model.trainable = True

    for layer in base_model.layers[:-30]:
        layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-5),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=10,
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=1
    )

    # -------- Evaluation --------
    loss, acc = model.evaluate(test_ds)

    # -------- Classification Report --------
    y_true = np.concatenate([y for x, y in test_ds], axis=0)
    y_pred = np.argmax(model.predict(test_ds), axis=1)

    report = classification_report(y_true, y_pred, output_dict=True)

    results[name] = {
        "Accuracy": acc,
        "Precision": report["weighted avg"]["precision"],
        "Recall": report["weighted avg"]["recall"],
        "F1-score": report["weighted avg"]["f1-score"]
    }

    model.save(f"{name}.keras")


🚀 Training MobileNetV2...

Epoch 1/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 39s 880ms/step - accuracy: 0.3912 - loss: 1.5653 - val_accuracy: 0.3375 - val_loss: 1.1860
Epoch 2/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 31s 836ms/step - accuracy: 0.4966 - loss: 1.2468 - val_accuracy: 0.4625 - val_loss: 1.1354
Epoch 3/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 32s 844ms/step - accuracy: 0.5476 - loss: 1.0784 - val_accuracy: 0.5250 - val_loss: 1.0388
Epoch 4/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 32s 834ms/step - accuracy: 0.5544 - loss: 1.1097 - val_accuracy: 0.6250 - val_loss: 0.8841
Epoch 5/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 31s 820ms/step - accuracy: 0.5867 - loss: 0.9559 - val_accuracy: 0.5500 - val_loss: 0.9728
Epoch 6/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 40s 822ms/step - accuracy: 0.5901 - loss: 1.0005 - val_accuracy: 0.5500 - val_loss: 0.9054
Epoch 7/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 31s 837ms/step - accuracy: 0.6259 - loss: 0.9072 - val_accuracy: 0.6125 - val_loss: 0.8445
Epoch 8/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 31s 834ms/step - accuracy: 0.61

4/5 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step

5/5 ━━━━━━━━━━━━━━━━━━━━ 19s 3s/step

🚀 Training EfficientNetB0...

Epoch 1/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 61s 1s/step - accuracy: 0.4881 - loss: 1.2909 - val_accuracy: 0.6750 - val_loss: 0.7778
Epoch 2/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 46s 1s/step - accuracy: 0.5680 - loss: 1.1199 - val_accuracy: 0.7125 - val_loss: 0.7183
Epoch 3/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 46s 1s/step - accuracy: 0.5833 - loss: 1.0914 - val_accuracy: 0.6750 - val_loss: 0.7897
Epoch 4/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 47s 1s/step - accuracy: 0.5680 - loss: 1.0088 - val_accuracy: 0.6500 - val_loss: 0.7694
Epoch 5/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 48s 1s/step - accuracy: 0.6071 - loss: 0.9490 - val_accuracy: 0.6000 - val_loss: 0.7538
Epoch 6/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 47s 1s/step - accuracy: 0.6224 - loss: 0.8621 - val_accuracy: 0.6375 - val_loss: 0.7739
Epoch 7/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 46s 1s/step - accuracy: 0.6327 - loss: 0.8768 - val_accuracy: 0.6375 - val_loss: 0.7673
Epoch 8/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 46s 1s/step 

KeyboardInterrupt: 

# new **way** model

In [11]:
models_to_train = {
    "EfficientNetV2B0": (EfficientNetV2B0(weights='imagenet', include_top=False, input_shape=(224,224,3)), effv2_pre),
    "ConvNeXtTiny": (ConvNeXtTiny(weights='imagenet', include_top=False, input_shape=(224,224,3)), conv_pre),
    "DenseNet121": (DenseNet121(weights='imagenet', include_top=False, input_shape=(224,224,3)), dense_pre),
    "MobileNetV3Small": (MobileNetV3Small(weights='imagenet', include_top=False, input_shape=(224,224,3)), mob3_pre)
}

24274472/24274472 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [12]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=2,
        min_lr=1e-6
    )
]

In [ ]:
results = {}

for name, (base_model, pre_fn) in models_to_train.items():

    print(f"\n🚀 Training {name}\n")

    model, base_model = build_model(base_model, pre_fn)

    # Stage 1
    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=8,
        class_weight=class_weights,
        callbacks=callbacks
    )

    # Fine-tune only strong models
    if name in ["EfficientNetV2B0", "ConvNeXtTiny"]:
        print(f"\n🔧 Fine-tuning {name}\n")

        base_model.trainable = True

        for layer in base_model.layers[:-20]:
            layer.trainable = False

        model.compile(
            optimizer=tf.keras.optimizers.Adam(1e-5),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )

        model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=5,
            class_weight=class_weights,
            callbacks=callbacks
        )

    # Evaluation
    loss, acc = model.evaluate(test_ds)

    y_true = np.concatenate([y for x, y in test_ds], axis=0)
    y_pred = np.argmax(model.predict(test_ds), axis=1)

    report = classification_report(y_true, y_pred, output_dict=True)

    results[name] = {
        "Accuracy": acc,
        "Precision": report["weighted avg"]["precision"],
        "Recall": report["weighted avg"]["recall"],
        "F1-score": report["weighted avg"]["f1-score"]
    }

    model.save(f"{name}.keras")


🚀 Training EfficientNetV2B0

Epoch 1/8
37/37 ━━━━━━━━━━━━━━━━━━━━ 75s 2s/step - accuracy: 0.2959 - loss: 1.5896 - val_accuracy: 0.2500 - val_loss: 1.2484 - learning_rate: 1.0000e-04
Epoch 2/8
37/37 ━━━━━━━━━━━━━━━━━━━━ 75s 1s/step - accuracy: 0.4354 - loss: 1.3649 - val_accuracy: 0.3625 - val_loss: 1.0839 - learning_rate: 1.0000e-04
Epoch 3/8
37/37 ━━━━━━━━━━━━━━━━━━━━ 49s 1s/step - accuracy: 0.5153 - loss: 1.0890 - val_accuracy: 0.6625 - val_loss: 0.9014 - learning_rate: 1.0000e-04
Epoch 4/8
37/37 ━━━━━━━━━━━━━━━━━━━━ 81s 1s/step - accuracy: 0.5646 - loss: 1.0238 - val_accuracy: 0.6875 - val_loss: 0.8300 - learning_rate: 1.0000e-04
Epoch 5/8
37/37 ━━━━━━━━━━━━━━━━━━━━ 48s 1s/step - accuracy: 0.5357 - loss: 1.0920 - val_accuracy: 0.6750 - val_loss: 0.8096 - learning_rate: 1.0000e-04
Epoch 6/8
37/37 ━━━━━━━━━━━━━━━━━━━━ 48s 1s/step - accuracy: 0.5680 - loss: 0.9740 - val_accuracy: 0.6375 - val_loss: 0.8101 - learning_rate: 1.0000e-04
Epoch 7/8
37/37 ━━━━━━━━━━━━━━━━━━━━ 49s 1s/step - a